# TCGA-BRCA Drug-Class Profile Review V1

This notebook is review-only. It reads the latest saved drug-class profile review outputs from disk,
regenerates review tables in `05-results`, and does not reread raw treatment tables, normalize drug names,
redefine treatment groups, freeze treatment arms, perform modeling, or make treatment recommendations.

This is a descriptive review and arm-freeze candidate audit only.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_drug_class_profile_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest drug-class profile v1 pointer not found: {latest_pointer_path}. '
        'Run script 24 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
group_summary_path = repo_root / latest_pointer['drug_class_profile_v1_group_summary_tsv']
confounder_coverage_path = repo_root / latest_pointer['drug_class_profile_v1_confounder_coverage_tsv']
value_composition_path = repo_root / latest_pointer['drug_class_profile_v1_value_composition_tsv']
arm_freeze_audit_path = repo_root / latest_pointer['drug_class_profile_v1_arm_freeze_candidate_audit_tsv']
summary_path = repo_root / latest_pointer['drug_class_profile_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [
    group_summary_path,
    confounder_coverage_path,
    value_composition_path,
    arm_freeze_audit_path,
    summary_path,
    run_log_path,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required drug-class profile artifact not found: {required_path}')

group_summary_df = read_tsv(group_summary_path)
confounder_coverage_df = read_tsv(confounder_coverage_path)
value_composition_df = read_tsv(value_composition_path)
arm_freeze_audit_df = read_tsv(arm_freeze_audit_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError('run_log.json does not report status == completed.')
if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if group_summary_df.empty:
    raise ValueError('drug_class_profile_v1_group_summary.tsv contains no rows.')

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

print(f"Run ID                    : {latest_pointer['drug_class_profile_v1_run_id']}")
print(f"Drug norm run ID          : {latest_pointer['drug_name_normalization_v1_run_id']}")
print(f"Grouping run ID           : {latest_pointer['patient_treatment_grouping_v1_run_id']}")
print(f"OS endpoint run ID        : {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Baseline analysis run ID  : {latest_pointer['baseline_analysis_v1_run_id']}")
print(f"Pointer                   : {latest_pointer_path}")

In [ ]:
review_group_summary_path = results_root / '131_drug_class_profile_v1_group_summary.tsv'
review_confounder_coverage_path = results_root / '132_drug_class_profile_v1_confounder_coverage.tsv'
review_value_composition_path = results_root / '133_drug_class_profile_v1_value_composition.tsv'
review_arm_freeze_audit_path = results_root / '134_drug_class_profile_v1_arm_freeze_candidate_audit.tsv'
review_summary_path = results_root / '135_drug_class_profile_v1_summary.tsv'

group_summary_df.to_csv(review_group_summary_path, sep='\t', index=False)
confounder_coverage_df.to_csv(review_confounder_coverage_path, sep='\t', index=False)
value_composition_df.to_csv(review_value_composition_path, sep='\t', index=False)
arm_freeze_audit_df.to_csv(review_arm_freeze_audit_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_group_summary_path}')
print(f'Saved: {review_confounder_coverage_path}')
print(f'Saved: {review_value_composition_path}')
print(f'Saved: {review_arm_freeze_audit_path}')
print(f'Saved: {review_summary_path}')

In [ ]:
print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation_df = pd.DataFrame(
    [{'check': key, 'value': str(value)} for key, value in run_log.get('validation', {}).items()]
)
print('\n=== Validation ===')
display(validation_df)

counts_df = pd.DataFrame(
    [{'metric': key, 'value': str(value)} for key, value in run_log.get('counts', {}).items()]
)
print('\n=== Key counts ===')
display(counts_df)

readiness_df = summary_df[summary_df['summary_section'] == 'readiness'].reset_index(drop=True)
print('\n=== Readiness ===')
display(readiness_df[['summary_metric', 'summary_value', 'notes']])

print('\n=== Full summary TSV ===')
display(summary_df)

In [ ]:
print('=== Group summary (bucket-level) ===')
display(group_summary_df)

print('\n=== Arm-freeze candidate audit ===')
display(arm_freeze_audit_df)

# Confounder coverage focus: 4 coverage-ready fields across buckets
coverage_focus_df = confounder_coverage_df[
    confounder_coverage_df['field_name'].isin(
        ['er_status_by_ihc', 'pr_status_by_ihc', 'her2_status_by_ihc', 'ajcc_pathologic_tumor_stage']
    )
    & confounder_coverage_df['review_group_type'].isin(
        ['all_treated', 'single_known_class_clean', 'single_known_class_plus_unknown',
         'multi_known_class', 'unknown_only', 'missing_like_only']
    )
].reset_index(drop=True)
print('\n=== Confounder coverage (4 key fields, bucket-level) ===')
display(coverage_focus_df[['review_group_type', 'field_name', 'row_count',
                            'non_missing_count', 'missing_like_fraction']])

# Age summaries by bucket
age_df = value_composition_df[
    (value_composition_df['field_name'] == 'age_at_diagnosis')
    & value_composition_df['review_group_type'].isin(
        ['all_treated', 'single_known_class_clean', 'single_known_class_plus_unknown',
         'multi_known_class', 'unknown_only', 'missing_like_only']
    )
].reset_index(drop=True)
print('\n=== Age summaries by bucket ===')
display(age_df[['review_group_type', 'group_patient_count', 'non_missing_count',
                'numeric_min', 'numeric_median', 'numeric_max']])